In [ ]:
"""
Network preprocessing (calculate OD)

GPU acceleration via RAPIDS (cudf, cugraph) is expected.
"""

In [ ]:
# ============================================================
# 0. Python version & package installation (optional for notebooks)
# ============================================================
# Python version: 3.12.9 | packaged by conda‑forge | (main, Feb 14 2025) [GCC 13.3.0]
import sys, subprocess

import os
import time
from collections import Counter

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from tqdm import tqdm


import cudf
import cugraph
os.chdir(os.getcwd())

DATA_DIR = "data"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, "cities"), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, "groupednodes"), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, "groupedlinks"), exist_ok=True)

In [ ]:
# ============================================================
# 1. Join population / links to cities
# ------------------------------------------------------------

# Load raw spatial & population files
links = gpd.read_file('raw data/2022-TM-KA-MR-LLV2 도로망(2021년 기준)/02. 링크/ad0022_2021_KA.shp')
nodes = gpd.read_file('raw data/2022-TM-KA-MR-LLV2 도로망(2021년 기준)/01. 노드/ad0102_2021_KA.shp')
cities = gpd.read_file('raw data/bnd_sigungu_00_2023_2Q.shp')
jgg_pop = pd.read_csv('raw data/2023년기준_2022년_인구총괄(총인구).txt', sep='^', header=None)

# Convert coordinates to EPSG:5179
links = links.to_crs(epsg=5179)
nodes = nodes.to_crs(epsg=5179)
cities = cities.to_crs(epsg=5179)

# Pre‑process JGG population table
jgg_pop.columns = ['year', 'jgg_id', 'pop_type', 'pop']
jgg_pop = jgg_pop[jgg_pop['pop_type'] == 'to_in_001']
jgg_pop['SIGUNGU_CD'] = jgg_pop['jgg_id'].apply(lambda x: str(x)[:5])

# Merge the aggregated population data with the cities GeoDataFrame
city_population = jgg_pop.groupby('SIGUNGU_CD')['pop'].sum().reset_index()
cities = cities.merge(city_population, left_on='SIGUNGU_CD', right_on='SIGUNGU_CD', how='left')

# Select midsize cities
midsize_cities = cities[(cities['pop'] >= 50000) & (cities['pop'] < 1000000)]
midsize_cities = midsize_cities[~midsize_cities['SIGUNGU_NM'].str.endswith(('구', '군'))]

# Map Korean city names to English
korean_to_english = {
    "동해시": "Donghae",
    "평택시": "Pyeongtaek",
    "밀양시": "Miryang",
    "순천시": "Suncheon",
    "의왕시": "Uiwang",
    "경산시": "Gyeongsan",
    "목포시": "Mokpo",
    "의정부시": "Uijeongbu",
    "당진시": "Dangjin",
    "구미시": "Gumi",
    "여수시": "Yeosu",
    "거제시": "Geoje",
    "이천시": "Icheon",
    "광양시": "Gwangyang",
    "문경시": "Mungyeong",
    "보령시": "Boryeong",
    "군산시": "Gunsan",
    "진주시": "Jinju",
    "광주시": "Gwangju",
    "오산시": "Osan",
    "군포시": "Gunpo",
    "서귀포시": "Seogwipo",
    "제주시": "Jeju",
    "김해시": "Gimhae",
    "양주시": "Yangju",
    "김천시": "Gimcheon",
    "동두천시": "Dongducheon",
    "제천시": "Jecheon",
    "과천시": "Gwacheon",
    "익산시": "Iksan",
    "논산시": "Nonsan",
    "서산시": "Seosan",
    "김제시": "Gimje",
    "영천시": "Yeongcheon",
    "경주시": "Gyeongju",
    "사천시": "Sacheon",
    "영주시": "Yeongju",
    "하남시": "Hanam",
    "세종시": "Sejong",
    "화성시": "Hwaseong",
    "시흥시": "Siheung",
    "상주시": "Sangju",
    "통영시": "Tongyeong",
    "여주시": "Yeoju",
    "나주시": "Naju",
    "김포시": "Gimpo",
    "광명시": "Gwangmyeong",
    "파주시": "Paju",
    "춘천시": "Chuncheon",
    "부천시": "Bucheon",
    "남원시": "Namwon",
    "남양주시": "Namyangju",
    "속초시": "Sokcho",
    "포천시": "Pocheon",
    "양산시": "Yangsan",
    "충주시": "Chungju",
    "정읍시": "Jeongeup",
    "구리시": "Guri",
    "강릉시": "Gangneung",
    "안성시": "Anseong",
    "원주시": "Wonju",
    "공주시": "Gongju",
    "안동시": "Andong",
    "삼척시": "Samcheok",
    "아산시": "Asan"
}

# Add English name column & save
midsize_cities["SIGUNGU_NM_EN"] = midsize_cities["SIGUNGU_NM"].map(korean_to_english)

# (≈20 s)


In [ ]:
# ============================================================
# 2. Group links & nodes by city & filter by LCC ratio
# ------------------------------------------------------------

# Spatial join and calculate link length
links_with_cities = gpd.sjoin(
    links, 
    midsize_cities[['SIGUNGU_NM', 'SIGUNGU_CD', 'geometry']], 
    how="left", predicate='within'
)
nodes_with_cities = gpd.sjoin(
    nodes, 
    midsize_cities[['SIGUNGU_NM', 'SIGUNGU_CD', 'geometry']], 
    how="left", predicate='within'
)

links_with_cities = links_with_cities[links_with_cities['SIGUNGU_CD'].notnull()]
nodes_with_cities = nodes_with_cities[nodes_with_cities['SIGUNGU_CD'].notnull()]

links_with_cities["LENGTH"] = links_with_cities.geometry.length

# Lists to store results
valid_links = []
valid_nodes = []

# Build city-level networks and check LCC ratio
for city_name, group in tqdm(links_with_cities.groupby('SIGUNGU_NM'), desc='filtering by LCC ratio'):
    G_cugraph = cugraph.Graph()
    edges_cudf = cudf.DataFrame.from_pandas(
        group[['UP_FROM_NO', 'UP_TO_NODE', 'LENGTH', 'LINK_ID', 'SIGUNGU_NM']]
    )
    
    G_cugraph.from_cudf_edgelist(
        edges_cudf, source='UP_FROM_NO', destination='UP_TO_NODE', edge_attr='LENGTH'
    )
    
    nodes_in_G = edges_cudf[['UP_FROM_NO', 'UP_TO_NODE']].melt(value_name='NODE_ID')['NODE_ID'].unique()

    # Connected components
    cc_df = cugraph.connected_components(G_cugraph)
    cc_size = cc_df.groupby('labels').size().reset_index(name='size')  # ✅ labels로 수정
    largest_cc = cc_size['size'].max()
    total_nodes = len(cc_df)
    lcc_ratio = largest_cc / total_nodes if total_nodes > 0 else 0

    if lcc_ratio >= 0.95:
        valid_links.append(group)

        # Filter nodes that exist in the city graph
        city_nodes = nodes_with_cities[nodes_with_cities['SIGUNGU_NM'] == city_name]
        city_nodes = city_nodes[city_nodes['NODE_ID'].isin(nodes_in_G.to_pandas())]
        valid_nodes.append(city_nodes)
    else:
        print(f"[DELETE] {city_name}: total={total_nodes}, largest={largest_cc}, ratio={lcc_ratio:.3f}")


# Merge results
links_with_cities = pd.concat(valid_links, ignore_index=True)
nodes_with_cities = pd.concat(valid_nodes, ignore_index=True)

# Save output
links_with_cities.to_file("data/links_with_cities.gpkg", driver="GPKG")
nodes_with_cities.to_file("data/nodes_with_cities.gpkg", driver="GPKG")

midsize_cities = midsize_cities[midsize_cities['SIGUNGU_NM'].isin(links_with_cities['SIGUNGU_NM'].unique())]
midsize_cities.to_file('data/midsize_cities_index.gpkg', driver='GPKG', index=False)

In [ ]:
# ============================================================
# 3. Calculate OD matrices (cuGraph SSSP)
# ------------------------------------------------------------
# 3.1 Ensure output directory exists
if not os.path.exists('data/groupedod/'):
    os.makedirs('data/groupedod/')

processing_city = 0

# Loop over each city
for city_name, group in links_with_cities.groupby('SIGUNGU_NM'):
    start_time = time.time()
    processing_city += 1

    # 3.Skip if OD already computed
    if os.path.exists(f'data/groupedod/{city_name}_od.csv'):
        print(f'{processing_city}/{len(midsize_cities)} {city_name} already processed, skipping')
        continue
    
    # Build city graph in cuGraph
    G_cugraph = cugraph.Graph()
    edges_cudf = cudf.DataFrame.from_pandas(group[['UP_FROM_NO', 'UP_TO_NODE', 'LENGTH', 'LINK_ID']])
    G_cugraph.from_cudf_edgelist(edges_cudf, source='UP_FROM_NO', destination='UP_TO_NODE', edge_attr='LENGTH')
    
    city_nodes = nodes_with_cities[nodes_with_cities['SIGUNGU_NM'] == city_name]
    city_nodes_list = city_nodes['NODE_ID'].to_list()

    
    # Compute all‑pairs shortest paths (SSSP per source)
    od_frames = []
    def shortest_path(src):
        return cugraph.sssp(G_cugraph, source=src, edge_attr='LENGTH')

    for src in tqdm(city_nodes_list, desc=f'Processing {city_name} {processing_city}/{len(midsize_cities)}'):
        od = shortest_path(src)
        frame = pd.DataFrame({'origin': src,
                              'destination': od['vertex'].to_numpy(),
                              'distance': od['distance'].to_numpy(),
                              'predecessor': od['predecessor'].to_numpy()})
        od_frames.append(frame)
    
    # Concatenate results & post‑process distances
    od_df = pd.concat(od_frames, ignore_index=True)
    od_df = od_df[od_df['origin'] != od_df['destination']]
    od_df['origin'] = od_df['origin'].astype(str)
    od_df['destination'] = od_df['destination'].astype(str)

    node_xy = city_nodes.set_index('NODE_ID')[['X', 'Y']]
    od_df['euclidean_distance'] = np.linalg.norm(node_xy.loc[od_df['origin']].values - node_xy.loc[od_df['destination']].values, axis=1)

    od_df['distance_ratio'] = od_df['distance'] / od_df['euclidean_distance']
    
    
    
    od_df['interval'] = (od_df['euclidean_distance'] // 100).round().astype(str)
    reached = od_df[od_df['predecessor'] != '-1']
    interval_mean = reached.groupby('interval')['distance_ratio'].mean()
    od_df = od_df.merge(interval_mean.rename('mean_ratio'), on='interval', how='left')
    od_df.loc[od_df['predecessor'] == '-1', 'distance'] = od_df['euclidean_distance'] * od_df['mean_ratio']
    od_df['distance_ratio'] = od_df['distance'] / od_df['euclidean_distance']
    od_df.drop(columns=['interval', 'mean_ratio'], inplace=True)

    # Save OD matrix
    od_df.to_csv(f'data/groupedod/{city_name}_od.csv', index=False)
    print(f'{processing_city}/{len(midsize_cities)} {city_name} processed in {time.time() - start_time:.1f} s')

4. Calculate centrality metrics

In [ ]:
# ============================================================
# 4. Calculate centrality metrics
# ------------------------------------------------------------
# (Original centrality block copied verbatim; only step headers added)

# Ensure output directories exist
if not os.path.exists('data/groupednodes/'):
    os.makedirs('data/groupednodes/')
if not os.path.exists('data/groupedlinks/'):
    os.makedirs('data/groupedlinks/')

# 4.2 Helper: reconstruct route from predecessor dictionary
def find_route(origin, destination, predecessors):
    route = [destination]
    while (origin, route[-1]) in predecessors:
        route.append(predecessors[(origin, route[-1])])
    return route[::-1]

# Iterate over cities and compute node/link metrics
od_files = nodes_with_cities.groupby('SIGUNGU_NM').count().sort_values('NODE_ID', ascending=True).reset_index()['SIGUNGU_NM'].tolist()
processing_city = 0

for city_name in od_files:
    start_time = time.time()
    processing_city += 1

    node_out = f'data/groupednodes/{city_name}_node_centrality.gpkg'
    link_out = f'data/groupedlinks/{city_name}_link_betweenness.gpkg'
    if os.path.exists(node_out) and os.path.exists(link_out):
        print(f'{processing_city}/{len(od_files)} {city_name} already processed, skipping')
        continue

    print(f'\nprocessing {city_name}: {processing_city}/{len(od_files)}')

    # 4.3.1 Load OD matrix for the city
    od_df = pd.read_csv(f'data/groupedod/{city_name}_od.csv')
    od_df['origin'] = od_df['origin'].astype(str)
    od_df['destination'] = od_df['destination'].astype(str)
    od_df['predecessor'] = od_df['predecessor'].astype(str)

    od_df_reached = od_df[od_df['predecessor'] != '-1']
    predecessors = od_df_reached.set_index(['origin', 'destination'])['predecessor'].to_dict()

    node_spi_counter = {'points': Counter()}
    nodes_counter = {'points': Counter()}
    edges_counter = {'points': Counter()}

    # Reconstruct routes in chunks to save memory
    chunk_size = 10_000_000
    num_chunks = len(od_df_reached) // chunk_size + 1
    for chunk in range(num_chunks):
        start_idx = chunk * chunk_size
        end_idx = min((chunk + 1) * chunk_size, len(od_df_reached))
        routes_cache = []
        for _, row in tqdm(od_df_reached.iloc[start_idx:end_idx].iterrows(),
                           desc=f'  Processing Chunk {chunk+1}/{num_chunks}', total=end_idx - start_idx):
            route = find_route(row['origin'], row['destination'], predecessors)
            routes_cache.append(route)
        for route in tqdm(routes_cache, desc='     node counting', total=len(routes_cache)):
            node_spi_counter['points'][route[0]] += (len(route)*(len(route)-1))/2
            for node in route:
                nodes_counter['points'][node] += 1
            for i in range(len(route)-1):
                edge = (route[i], route[i+1])
                edges_counter['points'][edge] += 1
        del routes_cache

    # Merge counters back to GeoDataFrames
    city_nodes = nodes_with_cities[nodes_with_cities['SIGUNGU_NM'] == city_name].copy()
    node_route_df = pd.DataFrame(nodes_counter['points'].items(), columns=['NODE_ID', 'betweenness'])
    node_spi_df = pd.DataFrame(node_spi_counter['points'].items(), columns=['NODE_ID', 'spi'])
    city_nodes = city_nodes.merge(node_route_df, on='NODE_ID', how='left')
    city_nodes = city_nodes.merge(node_spi_df, on='NODE_ID', how='left')

    city_links = links_with_cities[links_with_cities['SIGUNGU_NM'] == city_name].copy()
    edge_route_df = pd.DataFrame(edges_counter['points'].items(), columns=['edge', 'betweenness'])
    edge_route_df[['UP_FROM_NO', 'UP_TO_NODE']] = pd.DataFrame(edge_route_df['edge'].tolist(), index=edge_route_df.index)
    edge_route_df = edge_route_df.drop(columns=['edge'])
    edge_route_df['betweenness'] *= 2  # undirected graph adjustment
    city_links = city_links.merge(edge_route_df, on=['UP_FROM_NO', 'UP_TO_NODE'], how='left')

    # Closeness & straightness
    centrality_index = od_df.groupby('origin')[['distance', 'distance_ratio']].mean().reset_index()
    centrality_index.columns = ['NODE_ID', 'closeness', 'straightness']

    nodes_per_m2 = len(city_nodes) / city_links['LENGTH'].sum()
    od_df['pairs'] = ((od_df['distance']*nodes_per_m2)*((od_df['distance']*nodes_per_m2)-1))/2
    centrality_index['expectedpairs'] = od_df.groupby('origin')['pairs'].sum().reset_index()['pairs']

    city_nodes = city_nodes.merge(centrality_index, on='NODE_ID', how='left')

    # Select final columns and save
    city_nodes = city_nodes[['NODE_ID', 'SIGUNGU_NM', 'geometry', 'betweenness', 'spi', 'expectedpairs',
                             'closeness', 'straightness']] #each closeness, straightness, and betweenness is the mean distance, detour rate, and mediation of all od pairs
    city_links = city_links[['LINK_ID', 'UP_FROM_NO', 'UP_TO_NODE', 'LENGTH', 'geometry',
                             'SIGUNGU_NM', 'SIGUNGU_CD', 'betweenness']]

    city_nodes.to_file(node_out, driver='GPKG', index=False)
    city_links.to_file(link_out, driver='GPKG', index=False)

    print(f'{processing_city}/{len(od_files)} {city_name} processed in {time.time() - start_time:.1f} s')
